In [37]:
import pandas as pd

data = pd.read_csv('../data/AB_comparison.csv')

In [38]:
def split_items(text):
    if pd.isna(text) or not text:
        return []
    return [item.strip() for item in text.split('\n') if item.strip() and not item.endswith(':')]

def count_items_in_free_text(text):
    return len(split_items(text))

data['first_num_techniques'] = data['first_question_2_techniques'].apply(count_items_in_free_text)
data['first_num_intentions'] = data['first_question_4_intention'].apply(count_items_in_free_text)
data['first_num_consequences'] = data['first_question_5_consequences'].apply(count_items_in_free_text)
data['first_num_reactions'] = data['first_question_9_reaction'].apply(count_items_in_free_text)

data['repeated_num_techniques'] = data['repeated_question_2_techniques'].apply(count_items_in_free_text)
data['repeated_num_intentions'] = data['repeated_question_4_intention'].apply(count_items_in_free_text)
data['repeated_num_consequences'] = data['repeated_question_5_consequences'].apply(count_items_in_free_text)
data['repeated_num_reactions'] = data['repeated_question_9_reaction'].apply(count_items_in_free_text)

In [39]:
unique_texts_with_first_comment = data[~data['first_comment'].isna()]['id'].nunique()
unique_texts_with_repeated_comment = data[~data['repeated_comment'].isna()]['id'].nunique()
unique_texts_with_either_comment = data[~data[['first_comment', 'repeated_comment']].isna().all(axis=1)]['id'].nunique()
unique_texts_with_first_comment, unique_texts_with_repeated_comment, unique_texts_with_either_comment

(78, 100, 117)

In [40]:
def mean_of_each_item(items):
    all_items = []
    for text in items:
        items = split_items(text)
        all_items.extend(items)
    all_items = [len(item) for item in all_items]
    return sum(all_items) / len(all_items)

per_group_first = data.groupby('annotator_group').agg({
    'first_num_techniques': 'mean',
    'first_num_intentions': 'mean',
    'first_num_consequences': 'mean',
    'first_num_reactions': 'mean',
    'first_comment': lambda x: x.notna().mean()
}).reset_index()

overall_averages = pd.DataFrame({
    'annotator_group': ['overall'],
    'first_num_techniques': [data['first_num_techniques'].mean()],
    'first_num_intentions': [data['first_num_intentions'].mean()],
    'first_len_intentions': [data['first_question_4_intention'].str.len().mean()],
    'first_len_single_intentions': [mean_of_each_item(data['first_question_4_intention'])],
    'first_num_consequences': [data['first_num_consequences'].mean()],
    'first_len_consequences': [data['first_question_5_consequences'].str.len().mean()],
    'first_len_single_consequences': [mean_of_each_item(data['first_question_5_consequences'])],
    'first_num_reactions': [data['first_num_reactions'].mean()],
    'first_len_reactions': [data['first_question_9_reaction'].str.len().mean()],
    'first_len_single_reactions': [mean_of_each_item(data['first_question_9_reaction'])],
    'first_comment': [data['first_comment'].notna().mean()],
    'first_len_comment': [data['first_comment'].str.len().mean()]
})

per_group_first['first_len_intentions'] = data.groupby('annotator_group')['first_question_4_intention'].apply(lambda x: x.str.len().mean()).values
per_group_first['first_len_single_intentions'] = data.groupby('annotator_group')['first_question_4_intention'].apply(lambda x: mean_of_each_item(x)).values
per_group_first['first_len_consequences'] = data.groupby('annotator_group')['first_question_5_consequences'].apply(lambda x: x.str.len().mean()).values
per_group_first['first_len_single_consequences'] = data.groupby('annotator_group')['first_question_5_consequences'].apply(lambda x: mean_of_each_item(x)).values
per_group_first['first_len_reactions'] = data.groupby('annotator_group')['first_question_9_reaction'].apply(lambda x: x.str.len().mean()).values
per_group_first['first_len_single_reactions'] = data.groupby('annotator_group')['first_question_9_reaction'].apply(lambda x: mean_of_each_item(x)).values

per_group_first = pd.concat([per_group_first, overall_averages], ignore_index=True)
per_group_first

,annotator_group,first_num_techniques,first_num_intentions,first_num_consequences,first_num_reactions,first_comment,first_len_intentions,first_len_single_intentions,first_len_consequences,first_len_single_consequences,first_len_reactions,first_len_single_reactions,first_len_comment
0,komunikacja,1.773333,1.360000,1.853333,1.446667,0.226667,87.551181,53.769608,131.575000,55.892086,96.182692,45.239631,NaN
1,nastolatek,2.932886,1.194631,1.845638,1.174497,0.140940,75.188406,57.533708,84.834646,38.265455,75.398230,47.994286,NaN
2,nauczyciel,2.006667,1.620000,2.106667,1.326667,0.053333,70.394161,38.880658,86.152672,34.734177,54.464912,30.407035,NaN
3,psycholog,2.346667,1.620000,2.706667,1.340000,0.166667,91.000000,48.518519,135.903226,40.359606,83.861386,41.218905,NaN
4,rodzic,3.100000,2.266667,3.386667,1.773333,0.086667,95.626761,39.026471,128.870504,34.204724,79.495575,32.887218,NaN
5,overall,2.431242,1.612817,2.380507,1.412550,0.134846,83.920118,46.123344,113.282371,39.707796,77.403670,39.035917,175.455446


In [41]:
per_group_repeated = data.groupby('annotator_group').agg({
    'repeated_num_techniques': 'mean',
    'repeated_num_intentions': 'mean',
    'repeated_num_consequences': 'mean',
    'repeated_num_reactions': 'mean',
    'repeated_comment': lambda x: x.notna().mean()
}).reset_index()

overall_averages_repeated = pd.DataFrame({
    'annotator_group': ['overall'],
    'repeated_num_techniques': [data['repeated_num_techniques'].mean()],
    'repeated_num_intentions': [data['repeated_num_intentions'].mean()],
    'repeated_len_intentions': [data['repeated_question_4_intention'].str.len().mean()],
    'repeated_len_single_intentions': [mean_of_each_item(data['repeated_question_4_intention'])],
    'repeated_num_consequences': [data['repeated_num_consequences'].mean()],
    'repeated_len_consequences': [data['repeated_question_5_consequences'].str.len().mean()],
    'repeated_len_single_consequences': [mean_of_each_item(data['repeated_question_5_consequences'])],
    'repeated_num_reactions': [data['repeated_num_reactions'].mean()],
    'repeated_len_reactions': [data['repeated_question_9_reaction'].str.len().mean()],
    'repeated_len_single_reactions': [mean_of_each_item(data['repeated_question_9_reaction'])],
    'repeated_comment': [data['repeated_comment'].notna().mean()],
    'repeated_len_comment': [data['repeated_comment'].str.len().mean()]
})

per_group_repeated['repeated_len_intentions'] = data.groupby('annotator_group')['repeated_question_4_intention'].apply(lambda x: x.str.len().mean()).values
per_group_repeated['repeated_len_single_intentions'] = data.groupby('annotator_group')['repeated_question_4_intention'].apply(lambda x: mean_of_each_item(x)).values
per_group_repeated['repeated_len_consequences'] = data.groupby('annotator_group')['repeated_question_5_consequences'].apply(lambda x: x.str.len().mean()).values
per_group_repeated['repeated_len_single_consequences'] = data.groupby('annotator_group')['repeated_question_5_consequences'].apply(lambda x: mean_of_each_item(x)).values
per_group_repeated['repeated_len_reactions'] = data.groupby('annotator_group')['repeated_question_9_reaction'].apply(lambda x: x.str.len().mean()).values
per_group_repeated['repeated_len_single_reactions'] = data.groupby('annotator_group')['repeated_question_9_reaction'].apply(lambda x: mean_of_each_item(x)).values

per_group_repeated = pd.concat([per_group_repeated, overall_averages_repeated], ignore_index=True)
per_group_repeated

,annotator_group,repeated_num_techniques,repeated_num_intentions,repeated_num_consequences,repeated_num_reactions,repeated_comment,repeated_len_intentions,repeated_len_single_intentions,repeated_len_consequences,repeated_len_single_consequences,repeated_len_reactions,repeated_len_single_reactions,repeated_len_comment
0,komunikacja,1.953333,2.166667,2.933333,2.093333,0.366667,156.097744,62.827692,163.738806,48.843182,146.518868,47.477707,NaN
1,nastolatek,2.302013,1.557047,2.288591,1.496644,0.093960,82.195804,49.900862,89.397163,35.988270,78.218487,40.905830,NaN
2,nauczyciel,1.793333,1.566667,2.080000,1.286667,0.173333,86.755556,49.034043,86.686567,36.291667,63.049180,39.119171,NaN
3,psycholog,2.633333,1.646667,3.600000,1.566667,0.253333,93.827068,49.311741,142.085938,32.488889,93.829787,36.489362,NaN
4,rodzic,3.440000,1.726667,3.153333,1.473333,0.086667,100.506849,56.011583,155.814286,45.008457,104.513514,51.524887,NaN
5,overall,2.424566,1.732977,2.811749,1.583445,0.194927,103.449275,54.087827,127.271787,39.847578,95.927536,43.458685,127.876712


In [42]:
per_group = pd.merge(per_group_first, per_group_repeated, on='annotator_group')

cols = []
for col in per_group_first.columns:
    if col != 'annotator_group':
        cols.append(col)
        cols.append(col.replace('first', 'repeated'))
    else:
        cols.append(col)
per_group = per_group[cols]

for col in per_group.columns:
    if col != 'annotator_group':
        per_group[col] = per_group[col].round(2)

per_group.to_csv("../results/phase_2_AB_statistics_per_group.csv", index=False)
per_group

,annotator_group,first_num_techniques,repeated_num_techniques,first_num_intentions,repeated_num_intentions,first_num_consequences,repeated_num_consequences,first_num_reactions,repeated_num_reactions,first_comment,...,first_len_consequences,repeated_len_consequences,first_len_single_consequences,repeated_len_single_consequences,first_len_reactions,repeated_len_reactions,first_len_single_reactions,repeated_len_single_reactions,first_len_comment,repeated_len_comment
0,komunikacja,1.77,1.95,1.36,2.17,1.85,2.93,1.45,2.09,0.23,...,131.57,163.74,55.89,48.84,96.18,146.52,45.24,47.48,NaN,NaN
1,nastolatek,2.93,2.30,1.19,1.56,1.85,2.29,1.17,1.50,0.14,...,84.83,89.40,38.27,35.99,75.40,78.22,47.99,40.91,NaN,NaN
2,nauczyciel,2.01,1.79,1.62,1.57,2.11,2.08,1.33,1.29,0.05,...,86.15,86.69,34.73,36.29,54.46,63.05,30.41,39.12,NaN,NaN
3,psycholog,2.35,2.63,1.62,1.65,2.71,3.60,1.34,1.57,0.17,...,135.90,142.09,40.36,32.49,83.86,93.83,41.22,36.49,NaN,NaN
4,rodzic,3.10,3.44,2.27,1.73,3.39,3.15,1.77,1.47,0.09,...,128.87,155.81,34.20,45.01,79.50,104.51,32.89,51.52,NaN,NaN
5,overall,2.43,2.42,1.61,1.73,2.38,2.81,1.41,1.58,0.13,...,113.28,127.27,39.71,39.85,77.40,95.93,39.04,43.46,175.46,127.88


In [43]:
# average rows with "komunikacja" and "psycholog" into a new row "experts" and average rows "nastolatek", "nauczyciel" and "rodzic" into a new row "non-experts"
expert_groups = ['komunikacja', 'psycholog']
non_expert_groups = ['nastolatek', 'nauczyciel', 'rodzic']
expert_row = pd.DataFrame({
    'annotator_group': ['experts'],
    **{col: per_group[per_group['annotator_group'].isin(expert_groups)][col].mean() for col in per_group.columns if col != 'annotator_group'}
})
non_expert_row = pd.DataFrame({
    'annotator_group': ['non-experts'],
    **{col: per_group[per_group['annotator_group'].isin(non_expert_groups)][col].mean() for col in per_group.columns if col != 'annotator_group'}
})
per_group = pd.concat([per_group, expert_row, non_expert_row], ignore_index=True)
per_group

,annotator_group,first_num_techniques,repeated_num_techniques,first_num_intentions,repeated_num_intentions,first_num_consequences,repeated_num_consequences,first_num_reactions,repeated_num_reactions,first_comment,...,first_len_consequences,repeated_len_consequences,first_len_single_consequences,repeated_len_single_consequences,first_len_reactions,repeated_len_reactions,first_len_single_reactions,repeated_len_single_reactions,first_len_comment,repeated_len_comment
0,komunikacja,1.77,1.95,1.360000,2.17,1.85,2.930000,1.450000,2.09,0.230000,...,131.570,163.740000,55.890000,48.840000,96.180000,146.520000,45.240000,47.480,NaN,NaN
1,nastolatek,2.93,2.30,1.190000,1.56,1.85,2.290000,1.170000,1.50,0.140000,...,84.830,89.400000,38.270000,35.990000,75.400000,78.220000,47.990000,40.910,NaN,NaN
2,nauczyciel,2.01,1.79,1.620000,1.57,2.11,2.080000,1.330000,1.29,0.050000,...,86.150,86.690000,34.730000,36.290000,54.460000,63.050000,30.410000,39.120,NaN,NaN
3,psycholog,2.35,2.63,1.620000,1.65,2.71,3.600000,1.340000,1.57,0.170000,...,135.900,142.090000,40.360000,32.490000,83.860000,93.830000,41.220000,36.490,NaN,NaN
4,rodzic,3.10,3.44,2.270000,1.73,3.39,3.150000,1.770000,1.47,0.090000,...,128.870,155.810000,34.200000,45.010000,79.500000,104.510000,32.890000,51.520,NaN,NaN
5,overall,2.43,2.42,1.610000,1.73,2.38,2.810000,1.410000,1.58,0.130000,...,113.280,127.270000,39.710000,39.850000,77.400000,95.930000,39.040000,43.460,175.46,127.88
6,experts,2.06,2.29,1.490000,1.91,2.28,3.265000,1.395000,1.83,0.200000,...,133.735,152.915000,48.125000,40.665000,90.020000,120.175000,43.230000,41.985,NaN,NaN
7,non-experts,2.68,2.51,1.693333,1.62,2.45,2.506667,1.423333,1.42,0.093333,...,99.950,110.633333,35.733333,39.096667,69.786667,81.926667,37.096667,43.850,NaN,NaN


In [44]:
from scipy.stats import mannwhitneyu
import numpy as np

def calculate_significance_groups(data, group_map, col_pairs):
    results = {}
    for measure, first_col, repeated_col in col_pairs:
        results[measure] = {}
        for group_name, members in group_map.items():
            grp = data[data['annotator_group'].isin(members)]
            grp_valid = grp[[first_col, repeated_col]].dropna()
            if len(grp_valid) < 2:
                results[measure][group_name] = np.nan
                continue
            try:
                _, p_value = mannwhitneyu(
                    grp_valid[first_col],
                    grp_valid[repeated_col],
                    alternative='two-sided'
                )
            except Exception:
                p_value = np.nan
            results[measure][group_name] = p_value
    return results

group_map = {
    'experts': ['komunikacja', 'psycholog'],
    'non-experts': ['nastolatek', 'nauczyciel', 'rodzic']
}

col_pairs = [
    ('reactions', 'first_num_reactions', 'repeated_num_reactions'),
    ('intentions', 'first_num_intentions', 'repeated_num_intentions'),
    ('consequences', 'first_num_consequences', 'repeated_num_consequences'),
]

significance_results = calculate_significance_groups(data, group_map, col_pairs)
significance_results

{'reactions': {'experts': np.float64(0.046578639765500196),
  'non-experts': np.float64(0.6297202788052895)},
 'intentions': {'experts': np.float64(3.259336372233735e-05),
  'non-experts': np.float64(0.6742525512503728)},
 'consequences': {'experts': np.float64(9.897633540337569e-07),
  'non-experts': np.float64(0.015851004813072528)}}

In [45]:
column_list = ['question_1_influence_presence', 'question_3_intention_clarity', 'question_6_consequences_severity', 'question_7_submission', 'question_8_resistance']

difficult_to_assess_group = pd.DataFrame(per_group['annotator_group'])

for column in column_list:
    for prefix in ['first_', 'repeated_']:
        full_column = prefix + column
        grouped_data = data.groupby('annotator_group')[full_column].apply(lambda x: (x == 'Trudno ocenić').sum() / len(x))
        difficult_to_assess_group[full_column + '_trudno_ocenic'] = difficult_to_assess_group['annotator_group'].map(grouped_data)

        if difficult_to_assess_group[full_column + '_trudno_ocenic'].isna().any():
            overall_value = (data[full_column] == 'Trudno ocenić').sum() / len(data)
            difficult_to_assess_group.loc[difficult_to_assess_group['annotator_group'] == 'overall', full_column + '_trudno_ocenic'] = overall_value

for prefix in ['first_', 'repeated_']:
    difficult_to_assess_group[prefix + 'total_trudno_ocenic'] = difficult_to_assess_group[[col for col in difficult_to_assess_group.columns if col.startswith(prefix) and col.endswith('_trudno_ocenic')]].mean(axis=1)

difficult_to_assess_group

,annotator_group,first_question_1_influence_presence_trudno_ocenic,repeated_question_1_influence_presence_trudno_ocenic,first_question_3_intention_clarity_trudno_ocenic,repeated_question_3_intention_clarity_trudno_ocenic,first_question_6_consequences_severity_trudno_ocenic,repeated_question_6_consequences_severity_trudno_ocenic,first_question_7_submission_trudno_ocenic,repeated_question_7_submission_trudno_ocenic,first_question_8_resistance_trudno_ocenic,repeated_question_8_resistance_trudno_ocenic,first_total_trudno_ocenic,repeated_total_trudno_ocenic
0,komunikacja,0.033333,0.000000,0.020000,0.006667,0.020000,0.000000,0.333333,0.400000,0.126667,0.240000,0.106667,0.129333
1,nastolatek,0.020134,0.000000,0.006711,0.000000,0.026846,0.000000,0.328859,0.348993,0.255034,0.268456,0.127517,0.123490
2,nauczyciel,0.020000,0.033333,0.020000,0.006667,0.026667,0.033333,0.353333,0.420000,0.280000,0.253333,0.140000,0.149333
3,psycholog,0.033333,0.020000,0.013333,0.013333,0.033333,0.020000,0.373333,0.393333,0.206667,0.300000,0.132000,0.149333
4,rodzic,0.006667,0.000000,0.013333,0.000000,0.026667,0.000000,0.393333,0.406667,0.246667,0.266667,0.137333,0.134667
5,overall,0.022697,0.010681,0.014686,0.005340,0.026702,0.010681,0.356475,0.393858,0.222964,0.265688,0.128705,0.137250
6,experts,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,non-experts,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [46]:
certainty_per_group = pd.merge(
    data.groupby('annotator_group')['first_certainty_level'].mean().reset_index(),
    data.groupby('annotator_group')['repeated_certainty_level'].mean().reset_index(),
    on='annotator_group'
)

overall_certainty = pd.DataFrame({
    'annotator_group': ['overall'],
    'first_certainty_level': [data['first_certainty_level'].mean()],
    'repeated_certainty_level': [data['repeated_certainty_level'].mean()]
})

certainty_per_group = pd.concat([certainty_per_group, overall_certainty], ignore_index=True)
certainty_per_group

,annotator_group,first_certainty_level,repeated_certainty_level
0,komunikacja,3.393103,3.671141
1,nastolatek,3.717241,4.040268
2,nauczyciel,3.692308,3.798611
3,psycholog,3.950000,4.306667
4,rodzic,3.587413,3.789116
5,overall,3.666201,3.922869


In [47]:
# calculate certainty for experts and non-experts and add to the certainty_per_group dataframe
expert_certainty = pd.DataFrame({
    'annotator_group': ['experts'],
    'first_certainty_level': [data[data['annotator_group'].isin(expert_groups)]['first_certainty_level'].mean()],
    'repeated_certainty_level': [data[data['annotator_group'].isin(expert_groups)]['repeated_certainty_level'].mean()]
})
non_expert_certainty = pd.DataFrame({
    'annotator_group': ['non-experts'],
    'first_certainty_level': [data[data['annotator_group'].isin(non_expert_groups)]['first_certainty_level'].mean()],
    'repeated_certainty_level': [data[data['annotator_group'].isin(non_expert_groups)]['repeated_certainty_level'].mean()]
})
certainty_per_group = pd.concat([certainty_per_group, expert_certainty, non_expert_certainty], ignore_index=True)
certainty_per_group

,annotator_group,first_certainty_level,repeated_certainty_level
0,komunikacja,3.393103,3.671141
1,nastolatek,3.717241,4.040268
2,nauczyciel,3.692308,3.798611
3,psycholog,3.950000,4.306667
4,rodzic,3.587413,3.789116
5,overall,3.666201,3.922869
6,experts,3.666667,3.989967
7,non-experts,3.665893,3.877273


In [48]:
# add standard deviations to certainty_per_group dataframe
certainty_std_per_group = pd.merge(
    data.groupby('annotator_group')['first_certainty_level'].std().reset_index(),
    data.groupby('annotator_group')['repeated_certainty_level'].std().reset_index(),
    on='annotator_group'
)
overall_certainty_std = pd.DataFrame({
    'annotator_group': ['overall'],
    'first_certainty_level': [data['first_certainty_level'].std()],
    'repeated_certainty_level': [data['repeated_certainty_level'].std()]
})
certainty_std_per_group = pd.concat([certainty_std_per_group, overall_certainty_std], ignore_index=True)
certainty_per_group = pd.merge(certainty_per_group, certainty_std_per_group, on='annotator_group', suffixes=('_mean', '_std'))
certainty_per_group

,annotator_group,first_certainty_level_mean,repeated_certainty_level_mean,first_certainty_level_std,repeated_certainty_level_std
0,komunikacja,3.393103,3.671141,0.784366,0.817403
1,nastolatek,3.717241,4.040268,0.723643,0.676629
2,nauczyciel,3.692308,3.798611,0.714348,0.574272
3,psycholog,3.950000,4.306667,0.722954,0.704222
4,rodzic,3.587413,3.789116,0.585369,0.471536
5,overall,3.666201,3.922869,0.730190,0.697074


In [49]:
# calculate statistical significance of difference in certainty between first and repeated annotation for each group using mannwhitneyu test
def calculate_certainty_significance(data, group_map):
    results = {}
    for group_name, members in group_map.items():
        grp = data[data['annotator_group'].isin(members)]
        grp_valid = grp[['first_certainty_level', 'repeated_certainty_level']].dropna()
        if len(grp_valid) < 2:
            results[group_name] = np.nan
            continue
        try:
            _, p_value = mannwhitneyu(
                grp_valid['first_certainty_level'],
                grp_valid['repeated_certainty_level'],
                alternative='two-sided'
            )
        except Exception:
            p_value = np.nan
        results[group_name] = p_value
    return results

certainty_significance_results = calculate_certainty_significance(data, group_map)
certainty_significance_results

{'experts': np.float64(5.272391588426477e-06),
 'non-experts': np.float64(1.3529241021862447e-06)}

In [50]:
certainty_per_annotator = pd.merge(
    data.groupby('annotator')['first_certainty_level'].mean().reset_index(),
    data.groupby('annotator')['repeated_certainty_level'].mean().reset_index(),
    on='annotator'
)

certainty_per_annotator

,annotator,first_certainty_level,repeated_certainty_level
0,0,3.360000,3.310345
1,1,3.466667,3.600000
2,2,3.866667,4.200000
3,3,3.300000,3.700000
4,4,2.966667,3.533333
5,5,3.758621,3.700000
6,6,3.464286,3.933333
7,7,3.400000,3.758621
8,8,3.896552,3.833333
9,9,3.407407,3.714286
